# ⚙️ Actividad 09: Pipeline ETL Completo
---
**Entrada:** `data/02_interim/dataset_integrado.csv`  
**Salida:** `data/03_processed/master_dataset_fase1_v2.csv` + carga en PostgreSQL

Pasos:
1. Feature Engineering (codificación cíclica)
2. Escalamiento con StandardScaler
3. Carga en PostgreSQL (dim_tiempo → dim_ubicacion → fact_produccion_limon)
4. Exportación CSV final


In [1]:

import os, sys, json, warnings
import numpy as np, pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; PROCESSED=DIRS['processed']
SCALERS=DIRS['scalers']; PG_URI=CFG['PG_URI']
print(f"CWD: {os.getcwd()} | Config OK")


CWD: D:\CICLO 9\Machine-Learning-Multimodal--Agro-NLP-Clima- | Config OK


## 9.1 Extraer Dataset Integrado

In [2]:

df = pd.read_csv(f"{INTERIM}/dataset_integrado.csv")
print(f"Shape: {df.shape}")
print(f"Rango: {df['fecha_evento'].min()} → {df['fecha_evento'].max()}")
print(f"Columnas: {df.columns.tolist()}")
df.head(3)


Shape: (5880, 14)
Rango: 2021-01 → 2025-08
Columnas: ['fecha_evento', 'departamento', 'provincia', 'produccion_t', 'cosecha_ha', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'has_cultivo_perdidas', 'n_noticias', 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M']


,fecha_evento,departamento,provincia,produccion_t,cosecha_ha,precio_chacra_kg,num_emergencias,total_afectados,has_cultivo_perdidas,n_noticias,T2M,PRECTOTCORR,RH2M,WS2M
0,2021-01,AMAZONAS,BAGUA,25.7,0.0,1.960000,0.0,0.0,NaN,1.0,0.534199,-0.164969,0.005771,0.194035
1,2021-02,AMAZONAS,BAGUA,38.5,0.0,1.316667,0.0,0.0,NaN,0.0,0.717813,-0.236663,-0.445556,0.064450
2,2021-03,AMAZONAS,BAGUA,37.8,0.0,1.400000,0.0,0.0,NaN,3.0,0.421906,0.623674,0.416377,-0.116970


## 9.2 Feature Engineering — Codificación Cíclica

In [3]:

df['fecha_dt'] = pd.to_datetime(df['fecha_evento'])
df['anho']      = df['fecha_dt'].dt.year
df['mes']       = df['fecha_dt'].dt.month
df['trimestre'] = df['fecha_dt'].dt.quarter
df['month_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df = df.drop(columns=['fecha_dt'])
print("Codificación cíclica aplicada: month_sin, month_cos")
print(f"Trimestres: {sorted(df['trimestre'].unique())}")


Codificación cíclica aplicada: month_sin, month_cos
Trimestres: [np.int32(1), np.int32(2), np.int32(3), np.int32(4)]


## 9.3 Escalamiento con StandardScaler

In [4]:

FEATS = ['produccion_t','cosecha_ha','precio_chacra_kg',
         'num_emergencias','total_afectados','has_cultivo_perdidas','n_noticias',
         'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M']
cols = [c for c in FEATS if c in df.columns]

scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols].fillna(0))

scaler_path = f"{SCALERS}/scaler_fase1_v2.pkl"
joblib.dump(scaler, scaler_path)
print(f"Variables escaladas: {cols}")
print(f"Scaler guardado: {scaler_path}")


Variables escaladas: ['produccion_t', 'cosecha_ha', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'has_cultivo_perdidas', 'n_noticias', 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M']
Scaler guardado: models\scalers/scaler_fase1_v2.pkl


## 9.4 Carga en PostgreSQL

In [5]:

COORDS = {
    'AMAZONAS':(-6.23,-77.87),'ANCASH':(-9.53,-77.53),'APURIMAC':(-13.64,-72.88),
    'AREQUIPA':(-16.41,-71.54),'AYACUCHO':(-13.16,-74.22),'CAJAMARCA':(-7.16,-78.50),
    'CALLAO':(-12.06,-77.15),'CUSCO':(-13.53,-71.97),'HUANCAVELICA':(-12.78,-74.97),
    'HUANUCO':(-9.93,-76.24),'ICA':(-14.07,-75.73),'JUNIN':(-11.16,-75.00),
    'LA LIBERTAD':(-8.11,-79.03),'LAMBAYEQUE':(-6.77,-79.84),'LIMA':(-12.05,-77.04),
    'LORETO':(-3.75,-73.25),'MADRE DE DIOS':(-12.59,-69.19),'MOQUEGUA':(-17.19,-70.93),
    'PASCO':(-10.69,-76.26),'PIURA':(-5.19,-80.63),'PUNO':(-15.84,-70.02),
    'SAN MARTIN':(-6.52,-76.36),'TACNA':(-18.01,-70.25),'TUMBES':(-3.57,-80.45),
    'UCAYALI':(-8.38,-74.54),
}

try:
    engine = create_engine(PG_URI)
    with engine.connect() as conn:
        conn.execute(text("TRUNCATE TABLE fact_produccion_limon RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_tiempo RESTART IDENTITY CASCADE"))
        conn.execute(text("TRUNCATE TABLE dim_ubicacion RESTART IDENTITY CASCADE"))
        conn.commit()

    # dim_tiempo
    dim_t = df[['fecha_evento','anho','mes','trimestre','month_sin','month_cos']].drop_duplicates('fecha_evento')
    dim_t.to_sql('dim_tiempo', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_tiempo: {len(dim_t)} registros")

    # dim_ubicacion
    dim_u = df[['departamento','provincia']].drop_duplicates().reset_index(drop=True)
    dim_u['lat'] = dim_u['departamento'].map(lambda d: COORDS.get(d,(None,None))[0])
    dim_u['lon'] = dim_u['departamento'].map(lambda d: COORDS.get(d,(None,None))[1])
    dim_u.to_sql('dim_ubicacion', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] dim_ubicacion: {len(dim_u)} registros")

    # fact
    with engine.connect() as conn:
        dt_map = pd.read_sql('SELECT id_tiempo, fecha_evento FROM dim_tiempo', conn)
        du_map = pd.read_sql('SELECT id_ubicacion, departamento, provincia FROM dim_ubicacion', conn)

    df_f = df.merge(dt_map, on='fecha_evento').merge(du_map, on=['departamento','provincia'])
    fact_cols = ['id_tiempo','id_ubicacion','produccion_t','cosecha_ha','precio_chacra_kg',
                 'num_emergencias','total_afectados','n_noticias',
                 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M']
    fact_cols = [c for c in fact_cols if c in df_f.columns]
    df_load = df_f[fact_cols].dropna(subset=['id_tiempo','id_ubicacion'])
    df_load['id_tiempo'] = df_load['id_tiempo'].astype(int)
    df_load['id_ubicacion'] = df_load['id_ubicacion'].astype(int)
    df_load.to_sql('fact_produccion_limon', engine, if_exists='append', index=False, method='multi', chunksize=500)
    print(f"  [OK] fact_produccion_limon: {len(df_load):,} registros")
    engine.dispose()
except Exception as e:
    print(f"  [ERROR PostgreSQL] {e}")
    print("  Continuando sin carga en BD...")


  [ERROR PostgreSQL] 'utf-8' codec can't decode byte 0xf3 in position 85: invalid continuation byte
  Continuando sin carga en BD...


## 9.5 Exportar CSV Final

In [6]:

out = f"{PROCESSED}/master_dataset_fase1_v2.csv"
df.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Shape final: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
print(f"[OK] {out}")
print("\n[ACTIVIDAD 09] COMPLETADA.")


Shape final: (5880, 19)
Columnas: ['fecha_evento', 'departamento', 'provincia', 'produccion_t', 'cosecha_ha', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'has_cultivo_perdidas', 'n_noticias', 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M', 'anho', 'mes', 'trimestre', 'month_sin', 'month_cos']
[OK] data\03_processed/master_dataset_fase1_v2.csv

[ACTIVIDAD 09] COMPLETADA.


# Actividad 09 Finalizada OK
